## Logistic Regression Learning Curves

### Motivation and method

This analysis asks whether the selected L2 / `C=0.01` Logistic Regression and its `class_weight="balanced"` alternative still benefit from additional observations. It reads the protected ADA-ML-08 outputs; it does **not** rerun training. Within each shared five-fold split, only the training fold is stratified down to 5%, 10%, 25%, 50%, 75%, or 100%. The complete validation fold is retained. The reserved final test partition is never evaluated.

In [ ]:
learning_summary = pd.read_csv(project_root / "reports" / "tables" / "logistic_learning_curve_summary.csv")
learning_decision = pd.read_csv(project_root / "reports" / "tables" / "logistic_learning_curve_decision.csv")
display(learning_summary[["configuration_id", "train_fraction", "train_size_mean", "validation_roc_auc_mean", "validation_average_precision_mean", "roc_auc_generalization_gap", "fit_time_mean", "convergence_rate"]])
display(learning_decision)

### Figures

- [Train and validation ROC-AUC](../reports/figures/logistic_learning_curve_roc_auc.pdf)
- [Train and validation Average Precision](../reports/figures/logistic_learning_curve_average_precision.pdf)
- [Fit time by actual training size](../reports/figures/logistic_learning_curve_fit_time.pdf)
- [Validation threshold metrics](../reports/figures/logistic_learning_curve_threshold_metrics.pdf)

### Results, interpretation, limits, and conclusion

LR-LEARNING-ROC validation ROC-AUC is 0.839782, 0.849354, 0.855389, 0.857732, 0.858745, and 0.859201 from 5% through 100%; Average Precision is 0.457493, 0.481314, 0.498673, 0.504275, 0.506432, and 0.507592. LR-LEARNING-BALANCED ROC-AUC is 0.834234, 0.845888, 0.854096, 0.857267, 0.858418, and 0.859017; Average Precision is 0.443401, 0.472769, 0.494634, 0.502437, 0.504984, and 0.506454.

Both configurations improve through 100%, with diminishing returns. Their ROC-AUC gaps contract from 0.049999 to 0.002325 (unweighted) and 0.060521 to 0.002805 (balanced). Mean fit time rises from 0.148 to 3.084 seconds and from 0.444 to 4.733 seconds. All 60 fits converged. At 100%, balanced trades precision (0.284593 versus 0.691427) for recall (0.773666 versus 0.267758). The 75%→100% ROC/AP gains are only +0.000456/+0.001160 unweighted and +0.000599/+0.001470 balanced, supporting a late empirical plateau. Fold dispersion remains limited at 100%: ROC-AUC std is 0.003236 and 0.003122, while AP std is 0.008491 and 0.008518.

These are cross-validation estimates on one fixed training partition, not final-test performance. Fractions within a fold are reproducible stratified samples but are not guaranteed to be nested. Fit times depend on the current machine and workload. No threshold optimization or final model selection is performed here. The conclusion must therefore remain limited to observed data-volume sensitivity and whether late gains support an empirical plateau.

# 03 — Logistic Regression

**Objective:** Plan a reproducible linear baseline.  
**Owner:** TBD  
**Sprint:** TBD  

> Leakage warning: fit every preprocessing step inside training folds only.

## Naive Baselines

DummyClassifier baselines establish the performance expected without learned feature signal. The four registered strategies are `most_frequent` (`M01-DUMMY-001`), `prior` (`M01-DUMMY-002`), `stratified` (`M01-DUMMY-003`), and `uniform` (`M01-DUMMY-004`). This is important because the positive class represents approximately 10% of the training target: accuracy alone can therefore look high for a model that never detects a positive case.

The experiments were run by `scripts/run_dummy_baselines.py` with the shared cross-validation protocol on the training partition only. This notebook reads their recorded comparison; it does not rerun experiments, fit Logistic Regression, or access the final test partition.

## Logistic Regression L2 Baseline

**Scientific question:** does a regularized linear model learn discriminative signal beyond the naive baselines under the common training-only cross-validation protocol?

Experiment `M01-LR-001` uses `StandardScaler` followed by `LogisticRegression` in one Pipeline. Scaling is appropriate because coefficient optimization is sensitive to feature scales, and the Pipeline learns scaling parameters separately inside every training fold. L2 with `C=1.0` is a conventional, untuned baseline—not an optimality claim. Parameters are `penalty='l2'`, `solver='lbfgs'`, `class_weight=None`, `max_iter=1000`, `random_state=42`, `with_mean=True`, and `with_std=True`. Evaluation uses the shared five-fold stratified CV on the 160,000-row training partition only.

## Effect of Class Weighting

Because positives represent about 10% of training observations, `class_weight='balanced'` gives each class an inverse-frequency weight during Logistic Regression fitting. Experiment `M01-LR-002` is a controlled comparison with `M01-LR-001`: class weighting is the only changed factor; scaling, L2 penalty, `C=1.0`, solver, iteration limit, seed, folds, and metrics remain identical. The final test partition remains closed.

In [ ]:
class_weight_comparison = pd.read_csv(project_root / "reports" / "tables" / "logistic_class_weight_comparison.csv")
balanced_folds = pd.read_csv(project_root / "reports" / "experiments" / "M01-LR-002_fold_results.csv")
display(class_weight_comparison)
display(balanced_folds[["fold", "validation_roc_auc", "validation_average_precision", "validation_recall", "validation_f1"]])

### Interpretation and decision

Balanced weighting increases recall from 0.272484 to 0.773541 (+0.501057), balanced accuracy by 0.148786, and F1 by 0.025698. This comes with precision decreasing from 0.688813 to 0.284560 (−0.404253) and accuracy decreasing by 0.132687. Ranking performance is stable: ROC-AUC changes by −0.000176 and Average Precision by −0.001135. The balanced ROC-AUC gap is 0.002812, and fold scores remain stable. No convergence warning occurred.

`M01-LR-002` is therefore a recall-oriented alternative, not an unconditionally better model. With ROC-AUC as the primary ranking metric and no documented false-negative cost, retain `M01-LR-001` as the neutral baseline while preserving `M01-LR-002` for a future decision based on explicit operational costs. Figures `logistic_class_weight_metrics.pdf` and `logistic_class_weight_cv.pdf` visualize the trade-off and fold stability.

In [ ]:
logistic_comparison = pd.read_csv(project_root / "reports" / "tables" / "logistic_baseline_comparison.csv")
logistic_folds = pd.read_csv(project_root / "reports" / "experiments" / "M01-LR-001_fold_results.csv")
display(logistic_comparison)
display(logistic_folds[["fold", "train_roc_auc", "validation_roc_auc", "validation_average_precision"]])

### Results, convergence, and limitations

Mean validation ROC-AUC is 0.859188 ± 0.003239 and Average Precision is 0.507566 ± 0.008490. Compared with `most_frequent`, absolute improvements are 0.359188 ROC-AUC and 0.407078 Average Precision; relative Average Precision improvement is 4.0510 (405.10%). Balanced accuracy increases by 0.129343. Mean train ROC-AUC is 0.861525, giving a small train-minus-validation gap of 0.002337. The low fold ROC-AUC standard deviation indicates stable scores under these five folds. Mean fit time is 0.6450 seconds per fold, 3.739 times the recorded `most_frequent` fit time.

No `ConvergenceWarning` occurred with `lbfgs` and `max_iter=1000`. The model improves substantially over naive references but is not claimed to be optimal. No tuning, class balancing, threshold optimization, calibration, causal interpretation, or final-test evaluation was performed. Figures `logistic_vs_dummy_metrics.pdf` and `logistic_cv_scores.pdf` provide the recorded metric and fold comparisons.

In [ ]:
from pathlib import Path
import pandas as pd

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "configs" / "config.yaml").is_file())
comparison_path = project_root / "reports" / "tables" / "dummy_baseline_comparison.csv"
dummy_comparison = pd.read_csv(comparison_path)
dummy_comparison

### Interpretation

Interpret the recorded table jointly with `reports/figures/dummy_baseline_metrics.pdf`. The `most_frequent` and `prior` strategies both reach mean accuracy 0.8995 but balanced accuracy 0.5000, ROC-AUC 0.5000, and zero positive-class recall and F1. Their Average Precision is 0.1005, matching the positive prevalence. `stratified` and `uniform` change threshold-dependent metrics—especially accuracy, recall, and F1—but remain around chance discrimination (ROC-AUC 0.4995 and 0.5000). These results are reference floors, not learned models.

In [ ]:
from pathlib import Path

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "configs" / "config.yaml").is_file())
import sys
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.config import load_config

config = load_config()
print(config["project"]["name"])
print(config["project"]["random_state"])

## Logistic Regression Hyperparameter Search

**Scientific question:** which combination of regularization type, regularization strength, and class weighting provides the best cross-validated Logistic Regression behavior? Search `M01-LR-SEARCH-001` uses `GridSearchCV` because the predeclared space is small and discrete: L1/L2 × `C ∈ {0.01, 0.1, 1, 10, 100}` × `class_weight ∈ {None, balanced}`. This gives 20 candidates and 100 fits over the five shared stratified folds. Every candidate uses an unfitted `StandardScaler` → `LogisticRegression(solver='saga', max_iter=2000, random_state=42)` Pipeline, and ROC-AUC remains the predeclared refit metric.

The cells below read recorded candidate and decision tables; they never rerun the search or access the final test partition.

In [ ]:
grid_candidates = pd.read_csv(project_root / "reports" / "searches" / "M01-LR-SEARCH-001_candidates.csv")
grid_top = pd.read_csv(project_root / "reports" / "tables" / "logistic_grid_search_top_candidates.csv")
grid_decision = pd.read_csv(project_root / "reports" / "tables" / "logistic_grid_search_decision_table.csv")
display(grid_top)
display(grid_decision)

### Results, trade-offs, and limits

L2, `C=0.01`, unweighted (`candidate_002`) has the best ROC-AUC (0.859201). L1, `C=0.1`, unweighted (`candidate_005`) has the best Average Precision (0.507626). L2, `C=0.01`, balanced (`candidate_004`) has the best F1 (0.416119) and balanced accuracy (0.778194) at the default threshold, with recall 0.773666 but precision 0.284599. The ranking differences are tiny relative to fold variability, while class weighting drives the threshold-metric trade-off. Therefore no single final business configuration is selected.

The ROC-AUC figure shows performance across C with fold standard deviations; the trade-off figure contrasts precision, recall, F1, and balanced accuracy; the train/validation figure shows small ROC-AUC gaps (about 0.0023–0.0028 for the decision candidates), with no strong overfitting signal. Total search time was 974.66 seconds. Six `ConvergenceWarning` messages reported `max_iter=2000` being reached; parallel `GridSearchCV` output cannot reliably map them to candidates, results were not altered, and any corrective run requires a distinct Search ID. scikit-learn 1.8 also deprecates the explicit `penalty` API used by the predeclared assignment grid. The final test partition was not evaluated; threshold optimization, calibration, and definitive coefficient interpretation remain out of scope.

## Convergence, Sparsity and Coefficient Stability

The grid search emitted six convergence warnings that parallel execution could not attribute to candidates. This targeted audit therefore refits exactly `LR-SELECTED-ROC` (L2/C=0.01/unweighted), `LR-SELECTED-AP` (L1/C=0.1/unweighted), `LR-SELECTED-BALANCED` (L2/C=0.01/balanced), and `LR-L1-WEAK-REG` (L1/C=100/unweighted) on each shared training fold. With scikit-learn 1.8 it uses the recommended `l1_ratio=0` mapping for L2 and `l1_ratio=1` for L1; historical results are unchanged.

All 20 isolated fits converged without warning. Mean iteration counts were 19.6, 23.0, 31.4, and 21.6 respectively, far below `max_iter=2000`. Thus the previous warnings do not reproduce under the current API. Performance and convergence remain distinct evidence.

In [ ]:
convergence_audit = pd.read_csv(project_root / "reports" / "tables" / "logistic_convergence_audit.csv")
coefficient_stability = pd.read_csv(project_root / "reports" / "tables" / "logistic_coefficient_stability.csv")
l1_sparsity = pd.read_csv(project_root / "reports" / "tables" / "logistic_l1_sparsity_summary.csv")
model_selection = pd.read_csv(project_root / "reports" / "tables" / "logistic_model_selection_summary.csv")
display(model_selection)
display(l1_sparsity)

### Sparsity, stability, and provisional decision

L1/C=0.1 is only weakly sparse: folds retain 196–200 of 200 features, 193 appear in every fold, and the union is all 200. L1/C=100 retains all 200 features in every fold, so weak regularization provides no parsimony. For `LR-SELECTED-ROC`, the top standardized coefficients have consistent signs across all folds and small standard deviations; examples include `var_81`, `var_139`, `var_6`, `var_12`, and `var_76`. Coefficients are associations for standardized predictors, comparable in scale but not causal effects.

The audit strengthens `LR-SELECTED-ROC` as the provisional primary ranking configuration: it has the best predeclared ROC-AUC, converges fastest on average, and its leading coefficients are stable. `LR-SELECTED-AP` remains a marginal AP alternative but does not provide meaningful dimensional reduction. `LR-SELECTED-BALANCED` remains the recall-oriented threshold-dependent alternative. No test-final evaluation, threshold selection, or causal interpretation is made.

## Next step

Any further comparison or threshold decision must use an explicit scientific objective and a new experiment ID. The final test partition remains closed.